## 1. Установка необходимых библиотек

In [ ]:
!pip install --upgrade featuretools >> None

## 2. Импорт библиотек и настройка

In [ ]:
import pandas as pd

import featuretools as ft
from woodwork.logical_types import Categorical
from featuretools.primitives import IsWeekend, TimeSincePrevious, TimeSince, CumSum, CumMean, CumMin, CumMax

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

## 3. Загрузка и предобработка данных

In [ ]:
transaction_file = '/kaggle/input/alfa-challenge/df_transaction.pa'
df_transaction = pd.read_parquet(transaction_file)
df_transaction_copy = df_transaction.copy()
print("Данные успешно загружены.")

# Убираем ВОЗМОЖНЫЕ выбросы
df_transaction = df_transaction[df_transaction['amount'] > 0]

df_transaction['mcc_code'] = df_transaction['mcc_code'].astype(str)
df_transaction['merchant_name'] = df_transaction['merchant_name'].astype(str)
df_transaction['date_time'] = pd.to_datetime(df_transaction['date_time'])
print("Типы данных успешно преобразованы.")

## 4. Подготовка к автоматической генерации признаков

In [ ]:
clients = df_transaction[['client_num']].drop_duplicates()

es = ft.EntitySet(id="clients_data")

es = es.add_dataframe(
    dataframe=clients,
    dataframe_name="clients",
    index="client_num"
)

es = es.add_dataframe(
    dataframe=df_transaction,
    dataframe_name="transactions",
    make_index=True,
    index="transaction_id",
    time_index="date_time",
    logical_types={
        'mcc_code': Categorical,
        'merchant_name': Categorical
    }
)

es = es.add_relationship('clients', 'client_num', 'transactions', 'client_num')

## 5. Автоматическая генерация признаков

In [ ]:
print("Начало генерации признаков")

trans_primitives = ["day", "month", "year", "weekday", "hour", IsWeekend, TimeSincePrevious, TimeSince]
agg_primitives = ["sum", "mean", "std", "min", "max", "count", "num_unique", "mode", "skew", "median"]
groupby_trans_primitives = [CumSum, CumMean, CumMin, CumMax]

feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="clients",
    agg_primitives=agg_primitives,
    trans_primitives=trans_primitives,
    groupby_trans_primitives=groupby_trans_primitives,
    max_depth=250,
    verbose=True
)

print("Генерация признаков завершена.")
feature_matrix = feature_matrix.reset_index()
print(f"Количество сгенерированных признаков: {len(feature_defs)}")

## 6. Добавление минимальных и максимальных значений дня и месяца

In [ ]:
print("Добавление минимального и максимального дня и месяца для каждого клиента")

df_transaction['day'] = df_transaction['date_time'].dt.day
df_transaction['month'] = df_transaction['date_time'].dt.month

day_agg = df_transaction.groupby('client_num')['day'].agg(['min', 'max']).rename(columns={'min': 'min_day', 'max': 'max_day'}).reset_index()

month_agg = df_transaction.groupby('client_num')['month'].agg(['min', 'max']).rename(columns={'min': 'min_month', 'max': 'max_month'}).reset_index()

min_max_dates = pd.merge(day_agg, month_agg, on='client_num')

feature_matrix = feature_matrix.merge(min_max_dates, on='client_num', how='left')

print("Минимальные и максимальные день и месяц добавлены в датасет.")

## 7. Оценка важности MCC-кодов

In [ ]:
train_file = '/kaggle/input/alfa-challenge/train.pa'
df_train = pd.read_parquet(train_file)
df_transaction_copy = df_transaction_copy.merge(df_train, on='client_num', how='inner')
feature_matrix_copy = feature_matrix.copy()
feature_matrix_copy = feature_matrix_copy.merge(df_train, on='client_num', how='inner')

df_mcc_target = df_transaction_copy[['mcc_code', 'client_num']].merge(
    feature_matrix_copy[['client_num', 'target']], on='client_num', how='inner'
)
df_encoded = pd.get_dummies(df_mcc_target['mcc_code'], prefix='mcc')
df_encoded['target'] = df_mcc_target['target']

X = df_encoded.drop('target', axis=1)
y = df_encoded['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)

feature_importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({'mcc_code': X.columns, 'importance': feature_importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False).reset_index()

important_mcc = feature_importance_df[feature_importance_df['importance'] > 0.005]['mcc_code']
print(len(f'Количество важных mcc кодов: {important_mcc}'))
important_mcc_columns = important_mcc.tolist()

## 8. Генерация и объединение категориальных признаков

In [ ]:
# One-Hot Encoding только для значимых mcc_code
mcc_ohe = pd.get_dummies(df_transaction['mcc_code'], prefix='mcc')[important_mcc_columns]
df_transaction_ohe = df_transaction[['client_num']].join(mcc_ohe)

mcc_summed = df_transaction_ohe.groupby('client_num').sum().reset_index()

dataset_generated_with_cats = feature_matrix.merge(mcc_summed, on='client_num', how='left')

print("Датасет сгенерированных признаков с категориальными переменными создан.")

## 9. Сохранение данных

In [ ]:
output_file = "dataset_generated_with_cats.csv"
dataset_generated_with_cats.to_csv(output_file, index=False)